# RSNA Knee Abnormality Detection — 1,000-study single-run submission
## 46 true-gold + 954 report-supervised studies × 3 MRI series × 10 slices

This notebook trains and submits in one Kaggle rerun using the execution pattern already verified by the working baseline.

### Locked protocol

1. Split the 58 fully labelled studies into **46 development gold** and **12 untouched final gold**.
2. Train five KBioXLM multilabel report teachers on the 46 development reports and calculate out-of-fold teacher metrics.
3. Ensemble the five teachers over **954 disjoint non-gold reports**, retaining soft probabilities and fold disagreement.
4. Train the MRI model on the complete **1,000-study pool: 46 true-gold + 954 soft-labelled studies**.
5. Every study contributes exactly **3 series × 10 ordered slices** when three readable series exist.
6. Evaluate once on the untouched 12-study gold holdout.
7. Predict the substituted hidden-test MRIs and write `/kaggle/working/submission.csv`.

### Required input

Attach the competition data and the offline model input at:

```text
/kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1
```

Internet must remain disabled and a GPU accelerator should be enabled.


In [1]:
# Configuration and imports — no pip install and no network access
from pathlib import Path
import copy, gc, hashlib, json, math, os, random, time, warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import cv2
import pydicom
from torchvision.models import alexnet
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 2026
N_GOLD_DEV = 46
N_FINAL_HOLDOUT = 12
N_FOLDS = 5
N_WEAK_STUDIES = 954
SERIES_PER_STUDY = 3

MAX_TEXT_LENGTH = 512
TEXT_BATCH_SIZE = 4
PSEUDO_BATCH_SIZE = 16
TEXT_EPOCHS = 10
TEXT_PATIENCE = 3
TEXT_LR_ENCODER = 1e-5
TEXT_LR_HEAD = 2e-4
TEXT_WEIGHT_DECAY = 1e-2
UNFREEZE_LAST_N_TEXT_LAYERS = 2

IMAGE_SIZE = 224
MAX_SLICES = 10
SLICE_CHUNK = 16
IMAGE_EPOCHS = 4
IMAGE_PATIENCE = 2
IMAGE_LR_BACKBONE = 1e-5
IMAGE_LR_HEAD = 2e-4
IMAGE_WEIGHT_DECAY = 1e-5
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

# The quality gate warns by default so a submission is still produced.
MIN_TEACHER_MACRO_AUC = 0.55
STOP_IF_TEACHER_WEAK = False

LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

OUT = Path("/kaggle/working/rsna_kbioxlm_mrnet")
OUT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)
if AMP:
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 1. Validate exact offline input paths

Exact paths avoid slow recursive scans and eliminate ambiguity when other Kaggle inputs contain similarly named files. Hugging Face may store weights as `pytorch_model.bin`, sharded `.bin` files, or `model.safetensors`; `from_pretrained(..., local_files_only=True)` handles all supported formats.


In [2]:
# Exact Kaggle mount points. Change only these two lines if Kaggle displays a different slug/version.
ROOT_CANDIDATES = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
]
ROOT = next((path for path in ROOT_CANDIDATES if path.exists()), ROOT_CANDIDATES[0])
OFFLINE_MODELS = Path("/kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1")

KBIOXLM_DIR = OFFLINE_MODELS
ALEXNET_WEIGHTS = OFFLINE_MODELS / "alexnet-owt-7be5be79.pth"
TRAIN_SERIES_ROOT = ROOT / "train_series"

required_paths = {
    "competition root": ROOT,
    "train.csv": ROOT / "train.csv",
    "train_series.csv": ROOT / "train_series.csv",
    "sample_submission.csv": ROOT / "sample_submission.csv",
    "train_series directory": TRAIN_SERIES_ROOT,
    "offline model directory": OFFLINE_MODELS,
    "KBioXLM config": KBIOXLM_DIR / "config.json",
    "AlexNet checkpoint": ALEXNET_WEIGHTS,
}
missing = {name: str(path) for name, path in required_paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        "Required Kaggle inputs are missing. Confirm that both the competition data and "
        f"offline model input are attached. Missing: {missing}"
    )

weight_files = (
    list(KBIOXLM_DIR.glob("*.safetensors"))
    + list(KBIOXLM_DIR.glob("pytorch_model*.bin"))
    + list(KBIOXLM_DIR.glob("*.index.json"))
)
tokenizer_files = [
    KBIOXLM_DIR / "tokenizer.json",
    KBIOXLM_DIR / "sentencepiece.bpe.model",
    KBIOXLM_DIR / "tokenizer_config.json",
]
if not weight_files:
    raise FileNotFoundError(f"No Hugging Face model weights found in {KBIOXLM_DIR}")
if not any(path.exists() for path in tokenizer_files):
    raise FileNotFoundError(f"No tokenizer assets found in {KBIOXLM_DIR}")

train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
train_series = pd.read_csv(
    ROOT / "train_series.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
).drop_duplicates().copy()
sample_submission = pd.read_csv(ROOT / "sample_submission.csv", dtype={"StudyInstanceUID": str})

expected_columns = ["StudyInstanceUID"] + LABEL_COLS
if list(sample_submission.columns) != expected_columns:
    raise ValueError(
        "Unexpected submission schema. "
        f"Expected {expected_columns}, received {sample_submission.columns.tolist()}"
    )
if not train["StudyInstanceUID"].is_unique:
    raise ValueError("train.csv must contain one row per study")

print("Competition root:", ROOT)
print("KBioXLM directory:", KBIOXLM_DIR)
print("KBioXLM weight files:", [p.name for p in weight_files])
print("AlexNet checkpoint:", ALEXNET_WEIGHTS)
print("train:", train.shape, "train_series:", train_series.shape, "submission template:", sample_submission.shape)
display(sample_submission.head())


Competition root: /kaggle/input/competitions/rsna-knee-abnormality-detection
KBioXLM directory: /kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1
KBioXLM weight files: ['pytorch_model.bin']
AlexNet checkpoint: /kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1/alexnet-owt-7be5be79.pth
train: (4407, 14) train_series: (24371, 5) submission template: (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


## 2. Fixed, balanced 46/12 gold split

The split is deterministic from `SEED`. A bounded candidate search chooses 12 holdout studies whose label prevalence is close to the full gold set while giving absolute priority to preserving both classes in the 46-study teacher-development set. The resulting assignment is exported and never changed downstream.

In [3]:
gold_mask = train[LABEL_COLS].notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID", "Report"] + LABEL_COLS].copy().reset_index(drop=True)
non_gold = train.loc[~gold_mask].copy().reset_index(drop=True)

if len(gold) != N_GOLD_DEV + N_FINAL_HOLDOUT:
    raise RuntimeError(
        f"This locked protocol expects {N_GOLD_DEV + N_FINAL_HOLDOUT} fully labelled studies, found {len(gold)}. "
        "Review the competition data before changing the protocol."
    )

Y_gold = gold[LABEL_COLS].astype(int).to_numpy()

def choose_holdout(y, n_holdout, seed, trials=50000):
    rng = np.random.default_rng(seed)
    n = len(y)
    full_prev = y.mean(axis=0)
    best = None
    for _ in range(trials):
        h = np.sort(rng.choice(n, n_holdout, replace=False))
        keep = np.ones(n, dtype=bool); keep[h] = False
        yh, yd = y[h], y[keep]
        # Teacher-dev must retain both classes wherever the full gold set has both.
        missing_dev = sum(len(np.unique(yd[:, j])) < len(np.unique(y[:, j])) for j in range(y.shape[1]))
        # Holdout class coverage is desirable but may be impossible for very rare labels.
        missing_holdout = sum(len(np.unique(yh[:, j])) < 2 for j in range(y.shape[1]))
        prevalence_error = np.abs(yh.mean(axis=0) - full_prev).mean()
        score = 1000 * missing_dev + 0.25 * missing_holdout + prevalence_error
        if best is None or score < best[0]:
            best = (score, h, missing_dev, missing_holdout, prevalence_error)
    return best

split_score, hold_idx, miss_dev, miss_holdout, prev_error = choose_holdout(
    Y_gold, N_FINAL_HOLDOUT, SEED
)
hold_mask = np.zeros(len(gold), dtype=bool); hold_mask[hold_idx] = True
gold_dev = gold.loc[~hold_mask].copy().reset_index(drop=True)
gold_final = gold.loc[hold_mask].copy().reset_index(drop=True)

assert len(gold_dev) == N_GOLD_DEV and len(gold_final) == N_FINAL_HOLDOUT
assert set(gold_dev.StudyInstanceUID).isdisjoint(set(gold_final.StudyInstanceUID))

split_table = pd.concat([
    gold_dev[["StudyInstanceUID"]].assign(split="teacher_dev"),
    gold_final[["StudyInstanceUID"]].assign(split="final_gold_holdout"),
], ignore_index=True)
split_table.to_csv(OUT / "gold_split_46_12.csv", index=False)

prevalence = pd.DataFrame({
    "pathology": LABEL_COLS,
    "all_58": gold[LABEL_COLS].mean().values,
    "teacher_dev_46": gold_dev[LABEL_COLS].mean().values,
    "final_holdout_12": gold_final[LABEL_COLS].mean().values,
    "holdout_positive_count": gold_final[LABEL_COLS].sum().astype(int).values,
})
print("Gold:", len(gold), "teacher-dev:", len(gold_dev), "final holdout:", len(gold_final))
print("Candidate-search diagnostics:", {"missing_dev_classes": miss_dev, "missing_holdout_classes": miss_holdout, "prevalence_MAE": prev_error})
display(prevalence)

Gold: 58 teacher-dev: 46 final holdout: 12
Candidate-search diagnostics: {'missing_dev_classes': 0, 'missing_holdout_classes': 0, 'prevalence_MAE': np.float64(0.024664750957854406)}


,pathology,all_58,teacher_dev_46,final_holdout_12,holdout_positive_count
0,ACL,0.413793,0.413043,0.416667,5
1,MCL,0.155172,0.152174,0.166667,2
2,Medial Meniscus,0.448276,0.434783,0.500000,6
3,Lateral Meniscus,0.396552,0.391304,0.416667,5
4,Medial OA,0.258621,0.260870,0.250000,3
5,Lateral OA,0.189655,0.195652,0.166667,2
6,PF OA,0.362069,0.347826,0.416667,5
7,Effusion,0.603448,0.608696,0.583333,7
8,Synovitis,0.465517,0.456522,0.500000,6
9,Baker's,0.206897,0.217391,0.166667,2


## 3. Select three MRI series from 954 non-gold studies

The configured target remains 500 studies. If fewer eligible public studies are mounted, the notebook now uses every eligible study and emits a warning instead of crashing. Each selected study must still have three readable series, and all gold studies remain excluded.


In [4]:
def canonical_text(x):
    if pd.isna(x): return "UNKNOWN"
    return str(x).strip().upper()

def boolish_text(x):
    if pd.isna(x): return "UNKNOWN"
    t = str(x).strip().lower()
    if t in {"true", "t", "yes", "y", "1", "1.0"}: return "TRUE"
    if t in {"false", "f", "no", "n", "0", "0.0"}: return "FALSE"
    return t.upper()

series_meta = train_series.copy()
for col in ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]:
    if col not in series_meta.columns:
        series_meta[col] = "UNKNOWN"
series_meta["plane_norm"] = series_meta["Anatomical_Plane"].map(canonical_text).replace({"SAG":"SAGITTAL", "COR":"CORONAL", "AX":"AXIAL"})
series_meta["fluid_norm"] = series_meta["Fluid_Sensitive"].map(boolish_text)
series_meta["fat_norm"] = series_meta["Fat_Suppression"].map(boolish_text)
series_meta["signature"] = series_meta[["plane_norm", "fluid_norm", "fat_norm"]].agg("|".join, axis=1)

non_gold_reports = non_gold.loc[
    non_gold["Report"].notna() & non_gold["Report"].astype(str).str.strip().ne(""),
    ["StudyInstanceUID", "Report"],
].copy()
eligible = series_meta.merge(non_gold_reports, on="StudyInstanceUID", how="inner")

def series_dir_ok(row, root=TRAIN_SERIES_ROOT):
    d = root / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID)
    return d.exists() and any(p.is_file() and not p.name.startswith(".") for p in d.iterdir())

eligible["directory_ok"] = [series_dir_ok(r) for r in eligible.itertuples(index=False)]
eligible = eligible.loc[eligible.directory_ok].copy()

signature_counts = (
    eligible.groupby("signature")["StudyInstanceUID"].nunique()
    .sort_values(ascending=False)
)
TARGET_SIGNATURE = signature_counts.index[0]
signature_rank = {sig: rank for rank, sig in enumerate(signature_counts.index)}
eligible["signature_rank"] = eligible["signature"].map(signature_rank)

# Prefer the target signature, then other frequent signatures.
eligible["target_first"] = (eligible["signature"] != TARGET_SIGNATURE).astype(int)
eligible = eligible.sort_values(["target_first", "signature_rank", "StudyInstanceUID", "SeriesInstanceUID"])

# Require three distinct readable series, then choose 500 studies once.
series_counts = eligible.groupby("StudyInstanceUID")["SeriesInstanceUID"].nunique()
candidate_study_ids = np.array(sorted(series_counts[series_counts >= SERIES_PER_STUDY].index))
n_weak_actual = min(N_WEAK_STUDIES, len(candidate_study_ids))
if n_weak_actual == 0:
    raise RuntimeError("No non-gold studies have reports and three readable MRI series")
if n_weak_actual < N_WEAK_STUDIES:
    warnings.warn(
        f"Requested {N_WEAK_STUDIES} weak studies but only {n_weak_actual} are eligible; "
        "continuing with all eligible studies."
    )
rng = np.random.default_rng(SEED + 11)
selected_study_ids = np.sort(rng.choice(candidate_study_ids, n_weak_actual, replace=False))

weak_series_selection = (
    eligible[eligible.StudyInstanceUID.isin(selected_study_ids)]
    .drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])
    .groupby("StudyInstanceUID", group_keys=False)
    .head(SERIES_PER_STUDY)
    .sort_values(["StudyInstanceUID", "target_first", "signature_rank", "SeriesInstanceUID"])
    .reset_index(drop=True)
)
weak_studies = (
    non_gold_reports[non_gold_reports.StudyInstanceUID.isin(selected_study_ids)]
    .sort_values("StudyInstanceUID")
    .reset_index(drop=True)
)

assert len(weak_studies) == n_weak_actual
assert len(weak_series_selection) == n_weak_actual * SERIES_PER_STUDY
assert weak_series_selection.groupby("StudyInstanceUID").size().eq(SERIES_PER_STUDY).all()
assert set(weak_studies.StudyInstanceUID) == set(weak_series_selection.StudyInstanceUID)
assert set(weak_studies.StudyInstanceUID).isdisjoint(set(gold.StudyInstanceUID))

selection_cols = [
    "StudyInstanceUID", "SeriesInstanceUID", "Report", "Anatomical_Plane",
    "Fluid_Sensitive", "Fat_Suppression", "signature"
]
weak_series_selection[selection_cols].to_csv(OUT / "weak_500studies_3series_selection.csv", index=False)
print("Target signature:", TARGET_SIGNATURE)
print("Selected studies:", len(weak_studies))
print("Selected series:", len(weak_series_selection))
display(signature_counts.head(10).rename("eligible_studies").to_frame())
display(weak_series_selection[selection_cols].head(9))

Target signature: AXIAL|TRUE|TRUE
Selected studies: 954
Selected series: 2862


,eligible_studies
signature,
AXIAL|TRUE|TRUE,4349
SAGITTAL|FALSE|FALSE,4209
CORONAL|TRUE|TRUE,4192
SAGITTAL|TRUE|TRUE,4094
CORONAL|FALSE|FALSE,3361
AXIAL|FALSE|FALSE,844


,StudyInstanceUID,SeriesInstanceUID,Report,Anatomical_Plane,Fluid_Sensitive,Fat_Suppression,signature
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,Técnica: RMN de la rodilla. Resultados: Rotura...,Axial,1,1,AXIAL|TRUE|TRUE
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,Técnica: RMN de la rodilla. Resultados: Rotura...,Sagittal,0,0,SAGITTAL|FALSE|FALSE
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,Técnica: RMN de la rodilla. Resultados: Rotura...,Coronal,1,1,CORONAL|TRUE|TRUE
3,1.2.826.0.1.3680043.8.498.10025765742726180988...,1.2.826.0.1.3680043.8.498.52503045794545563216...,"MRI of left Knee with \n-Locator, SG PD FatSat...",Axial,1,1,AXIAL|TRUE|TRUE
4,1.2.826.0.1.3680043.8.498.10025765742726180988...,1.2.826.0.1.3680043.8.498.12296387404690065979...,"MRI of left Knee with \n-Locator, SG PD FatSat...",Sagittal,0,0,SAGITTAL|FALSE|FALSE
5,1.2.826.0.1.3680043.8.498.10025765742726180988...,1.2.826.0.1.3680043.8.498.58180411352017518670...,"MRI of left Knee with \n-Locator, SG PD FatSat...",Coronal,1,1,CORONAL|TRUE|TRUE
6,1.2.826.0.1.3680043.8.498.10029168398037940681...,1.2.826.0.1.3680043.8.498.93204062869632903043...,Antecedentes Clínicos:\nSíndrome meniscal rodi...,Axial,1,1,AXIAL|TRUE|TRUE
7,1.2.826.0.1.3680043.8.498.10029168398037940681...,1.2.826.0.1.3680043.8.498.61060692464231753798...,Antecedentes Clínicos:\nSíndrome meniscal rodi...,Sagittal,0,0,SAGITTAL|FALSE|FALSE
8,1.2.826.0.1.3680043.8.498.10029168398037940681...,1.2.826.0.1.3680043.8.498.10398122283136151623...,Antecedentes Clínicos:\nSíndrome meniscal rodi...,Coronal,1,1,CORONAL|TRUE|TRUE


## 4. Balanced five-fold assignment on the 46 teacher-development reports

This dependency-free search balances label prevalence while keeping fold sizes at 9–10. Missing validation classes are penalized when the global support makes coverage feasible. The assignment is study-level and exported.

In [5]:
def make_balanced_folds(y, n_splits, seed, trials=30000):
    rng = np.random.default_rng(seed)
    n = len(y)
    sizes = np.full(n_splits, n // n_splits, dtype=int)
    sizes[: n % n_splits] += 1
    global_prev = y.mean(axis=0)
    global_pos = y.sum(axis=0)
    best = None
    for _ in range(trials):
        perm = rng.permutation(n)
        fold_id = np.empty(n, dtype=int)
        start = 0
        for f, size in enumerate(sizes):
            fold_id[perm[start:start + size]] = f
            start += size
        score = 0.0
        missing = 0
        for f in range(n_splits):
            yf = y[fold_id == f]
            score += np.square(yf.mean(axis=0) - global_prev).mean()
            for j in range(y.shape[1]):
                if global_pos[j] >= n_splits and (len(y) - global_pos[j]) >= n_splits:
                    missing += int(len(np.unique(yf[:, j])) < 2)
        score += 0.2 * missing
        if best is None or score < best[0]:
            best = (score, fold_id.copy(), missing)
    return best

Y_dev = gold_dev[LABEL_COLS].astype(np.float32).to_numpy()
fold_score, dev_fold, fold_missing = make_balanced_folds(Y_dev, N_FOLDS, SEED + 23)
gold_dev["fold"] = dev_fold
gold_dev[["StudyInstanceUID", "fold"]].to_csv(OUT / "teacher_cv_folds.csv", index=False)

fold_summary = []
for f in range(N_FOLDS):
    idx = dev_fold == f
    row = {"fold": f, "n": int(idx.sum())}
    row.update({f"pos__{lab}": int(Y_dev[idx, j].sum()) for j, lab in enumerate(LABEL_COLS)})
    fold_summary.append(row)
print("Fold-search score:", fold_score, "feasible missing-class penalties:", fold_missing)
display(pd.DataFrame(fold_summary))

Fold-search score: 0.036051914 feasible missing-class penalties: 0


,fold,n,pos__ACL,pos__MCL,pos__Medial Meniscus,pos__Lateral Meniscus,pos__Medial OA,pos__Lateral OA,pos__PF OA,pos__Effusion,pos__Synovitis,pos__Baker's,pos__Contusion,pos__Fracture
0,0,10,4,1,5,3,3,3,4,6,4,3,3,2
1,1,9,4,1,3,4,2,2,2,6,5,2,3,3
2,2,9,4,1,5,3,3,1,2,6,3,2,2,4
3,3,9,5,1,4,4,2,2,5,6,5,2,4,2
4,4,9,2,3,3,4,2,1,3,4,4,1,3,3


## 5. KBioXLM multilabel report teacher

The original token-classification head in the uploaded checkpoint is discarded. We use the KBioXLM XLM-R encoder's `<s>` representation, dropout, and a fresh 12-output linear head. Most encoder parameters remain frozen; only the last two transformer blocks and the classification head train.

Each fold's best validation-loss state immediately predicts its held-out reports and all 500 pseudo-label reports. The five full checkpoints are therefore not retained, avoiding several gigabytes of redundant output while preserving the true five-model ensemble predictions.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(str(KBIOXLM_DIR), local_files_only=True, use_fast=True)

class ReportDataset(Dataset):
    def __init__(self, frame, with_targets):
        self.frame = frame.reset_index(drop=True)
        self.with_targets = with_targets
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        enc = tokenizer(
            str(row["Report"]),
            truncation=True,
            max_length=MAX_TEXT_LENGTH,
            padding=False,
        )
        item = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        if self.with_targets:
            item["labels"] = row[LABEL_COLS].astype(float).to_numpy(np.float32)
        item["row_index"] = int(row["row_index"])
        return item

def collate_reports(batch):
    padded = tokenizer.pad(
        [{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in batch],
        padding=True,
        return_tensors="pt",
    )
    out = dict(padded)
    out["row_index"] = torch.tensor([x["row_index"] for x in batch], dtype=torch.long)
    if "labels" in batch[0]:
        out["labels"] = torch.tensor(np.stack([x["labels"] for x in batch]), dtype=torch.float32)
    return out

class KBioXLM12(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(str(KBIOXLM_DIR), local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Linear(hidden, len(LABEL_COLS))
        for p in self.encoder.parameters():
            p.requires_grad = False
        layers = self.encoder.encoder.layer
        for layer in layers[-UNFREEZE_LAST_N_TEXT_LAYERS:]:
            for p in layer.parameters():
                p.requires_grad = True
        # Train the final encoder norm if this implementation exposes one.
        if hasattr(self.encoder.encoder, "layer_norm"):
            for p in self.encoder.encoder.layer_norm.parameters():
                p.requires_grad = True
    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.head(self.dropout(h))

def make_text_loaders(train_frame, val_frame):
    tr = train_frame.reset_index(drop=False).rename(columns={"index": "original_index"})
    va = val_frame.reset_index(drop=False).rename(columns={"index": "original_index"})
    tr["row_index"] = np.arange(len(tr))
    va["row_index"] = np.arange(len(va))
    return (
        DataLoader(ReportDataset(tr, True), batch_size=TEXT_BATCH_SIZE, shuffle=True,
                   num_workers=NUM_WORKERS, pin_memory=AMP, collate_fn=collate_reports),
        DataLoader(ReportDataset(va, True), batch_size=TEXT_BATCH_SIZE, shuffle=False,
                   num_workers=NUM_WORKERS, pin_memory=AMP, collate_fn=collate_reports),
        va,
    )

def predict_report_loader(model, loader, n_rows):
    model.eval()
    pred = np.zeros((n_rows, len(LABEL_COLS)), dtype=np.float32)
    with torch.inference_mode():
        for batch in loader:
            idx = batch.pop("row_index").numpy()
            batch.pop("labels", None)
            inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=AMP):
                logits = model(**inputs)
            pred[idx] = torch.sigmoid(logits).float().cpu().numpy()
    return pred

In [7]:
oof_pred = np.full((len(gold_dev), len(LABEL_COLS)), np.nan, dtype=np.float32)
pseudo_sum = np.zeros((len(weak_studies), len(LABEL_COLS)), dtype=np.float64)
pseudo_sumsq = np.zeros_like(pseudo_sum)
teacher_history = []

pseudo_frame = weak_studies[["StudyInstanceUID", "Report"]].copy().reset_index(drop=True)
pseudo_frame["row_index"] = np.arange(len(pseudo_frame))
pseudo_loader = DataLoader(
    ReportDataset(pseudo_frame, False), batch_size=PSEUDO_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=AMP, collate_fn=collate_reports
)

for fold in range(N_FOLDS):
    print(f"\n{'=' * 28} TEXT FOLD {fold + 1}/{N_FOLDS} {'=' * 28}")
    seed_everything(SEED + 100 + fold)
    tr_df = gold_dev.loc[gold_dev.fold != fold].copy()
    va_df = gold_dev.loc[gold_dev.fold == fold].copy()
    tr_loader, va_loader, va_indexed = make_text_loaders(tr_df, va_df)

    model = KBioXLM12().to(DEVICE)
    encoder_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": TEXT_LR_ENCODER},
        {"params": model.head.parameters(), "lr": TEXT_LR_HEAD},
    ], weight_decay=TEXT_WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    criterion = nn.BCEWithLogitsLoss()
    best_loss = math.inf
    best_state = None
    stale = 0

    for epoch in range(1, TEXT_EPOCHS + 1):
        model.train()
        train_losses = []
        for batch in tr_loader:
            batch.pop("row_index")
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                logits = model(**inputs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update()
            train_losses.append(float(loss.detach().cpu()))

        model.eval(); val_losses = []
        with torch.inference_mode():
            for batch in va_loader:
                batch.pop("row_index")
                labels = batch.pop("labels").to(DEVICE, non_blocking=True)
                inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=AMP):
                    loss = criterion(model(**inputs), labels)
                val_losses.append(float(loss.cpu()))
        train_loss = float(np.mean(train_losses)); val_loss = float(np.mean(val_losses))
        teacher_history.append({"fold": fold, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= TEXT_PATIENCE:
                break

    model.load_state_dict(best_state)
    # OOF loader indexes rows locally; map them back to gold_dev indices.
    local_val_pred = predict_report_loader(model, va_loader, len(va_indexed))
    original_indices = va_indexed["original_index"].to_numpy()
    oof_pred[original_indices] = local_val_pred

    fold_pseudo = predict_report_loader(model, pseudo_loader, len(pseudo_frame))
    pseudo_sum += fold_pseudo
    pseudo_sumsq += np.square(fold_pseudo)

    del model, optimizer, scaler, best_state
    gc.collect()
    if AMP: torch.cuda.empty_cache()

pd.DataFrame(teacher_history).to_csv(OUT / "teacher_training_history.csv", index=False)
assert np.isfinite(oof_pred).all(), "Every development report must have one OOF prediction"


============================ TEXT FOLD 1/5 ============================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: /kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 01 | train 0.6856 | val 0.6744
epoch 02 | train 0.6407 | val 0.6582
epoch 03 | train 0.6123 | val 0.6532
epoch 04 | train 0.5902 | val 0.6598
epoch 05 | train 0.5871 | val 0.6676
epoch 06 | train 0.5642 | val 0.6752

============================ TEXT FOLD 2/5 ============================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

XLMRobertaModel LOAD REPORT from: /kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 01 | train 0.6951 | val 0.6646
epoch 02 | train 0.6434 | val 0.6160
epoch 03 | train 0.6098 | val 0.5870
epoch 04 | train 0.6009 | val 0.5766
epoch 05 | train 0.5846 | val 0.5639
epoch 06 | train 0.5749 | val 0.5603
epoch 07 | train 0.5419 | val 0.5560
epoch 08 | train 0.5453 | val 0.5587
epoch 09 | train 0.5419 | val 0.5588
epoch 10 | train 0.5331 | val 0.5649

============================ TEXT FOLD 5/5 ============================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: /kaggle/input/models/amritgopinath/rsna-offline-models/pytorch/default/1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 01 | train 0.6737 | val 0.6548
epoch 02 | train 0.6314 | val 0.6410
epoch 03 | train 0.6257 | val 0.6391
epoch 04 | train 0.5936 | val 0.6415
epoch 05 | train 0.5829 | val 0.6428
epoch 06 | train 0.5844 | val 0.6415


## 6. OOF report-teacher evaluation and 5-model soft pseudo-labels

ROC-AUC and PR-AUC are omitted for a pathology when the 46 development labels contain only one class. Macro values are NaN-safe means across defined pathologies. F1 uses the fixed 0.5 threshold and is descriptive, not tuned on the final gold holdout.

In [8]:
def multilabel_metrics(y, p):
    rows = []
    for j, label in enumerate(LABEL_COLS):
        yt, pt = y[:, j].astype(int), p[:, j]
        two_classes = len(np.unique(yt)) == 2
        rows.append({
            "pathology": label,
            "n": len(yt),
            "positives": int(yt.sum()),
            "roc_auc": roc_auc_score(yt, pt) if two_classes else np.nan,
            "pr_auc": average_precision_score(yt, pt) if yt.sum() > 0 else np.nan,
            "f1_at_0.5": f1_score(yt, pt >= 0.5, zero_division=0),
        })
    metrics = pd.DataFrame(rows)
    macro = {
        "pathology": "MACRO",
        "n": int(metrics.n.sum()),
        "positives": int(metrics.positives.sum()),
        "roc_auc": metrics.roc_auc.mean(skipna=True),
        "pr_auc": metrics.pr_auc.mean(skipna=True),
        "f1_at_0.5": metrics["f1_at_0.5"].mean(skipna=True),
    }
    return pd.concat([metrics, pd.DataFrame([macro])], ignore_index=True)

teacher_metrics = multilabel_metrics(Y_dev, oof_pred)
teacher_metrics.to_csv(OUT / "teacher_oof_metrics.csv", index=False)
oof_table = gold_dev[["StudyInstanceUID", "fold"] + LABEL_COLS].copy()
for j, label in enumerate(LABEL_COLS):
    oof_table[f"oof__{label}"] = oof_pred[:, j]
oof_table.to_csv(OUT / "teacher_oof_predictions.csv", index=False)
display(teacher_metrics)

teacher_macro_auc = float(teacher_metrics.loc[teacher_metrics.pathology == "MACRO", "roc_auc"].iloc[0])
if not np.isfinite(teacher_macro_auc) or teacher_macro_auc < MIN_TEACHER_MACRO_AUC:
    msg = f"Teacher macro OOF ROC-AUC {teacher_macro_auc:.3f} is below the quality gate {MIN_TEACHER_MACRO_AUC:.2f}."
    if STOP_IF_TEACHER_WEAK:
        raise RuntimeError(msg)
    print("WARNING:", msg, "Continuing only because STOP_IF_TEACHER_WEAK=False.")

pseudo_mean = (pseudo_sum / N_FOLDS).astype(np.float32)
pseudo_var = np.maximum(pseudo_sumsq / N_FOLDS - np.square(pseudo_mean), 0.0)
pseudo_std = np.sqrt(pseudo_var).astype(np.float32)

weak_labels = weak_studies[["StudyInstanceUID", "Report"]].copy()
for j, label in enumerate(LABEL_COLS):
    weak_labels[f"weak__{label}"] = pseudo_mean[:, j]
    weak_labels[f"std__{label}"] = pseudo_std[:, j]
weak_labels.to_csv(OUT / "kbioxlm_500study_soft_labels.csv", index=False)

print("Pseudo-label probability summary:")
display(pd.DataFrame(pseudo_mean, columns=LABEL_COLS).describe().T[["mean", "std", "min", "max"]])
print("Mean five-fold disagreement:")
display(pd.Series(pseudo_std.mean(axis=0), index=LABEL_COLS, name="mean_fold_std").to_frame())

,pathology,n,positives,roc_auc,pr_auc,f1_at_0.5
0,ACL,46,19,0.389864,0.373118,0.000000
1,MCL,46,7,0.479853,0.153133,0.000000
2,Medial Meniscus,46,20,0.586538,0.577600,0.230769
3,Lateral Meniscus,46,18,0.427579,0.344984,0.000000
4,Medial OA,46,12,0.664216,0.553396,0.000000
5,Lateral OA,46,9,0.566066,0.231353,0.000000
6,PF OA,46,16,0.476042,0.430136,0.000000
7,Effusion,46,28,0.438492,0.566562,0.739726
8,Synovitis,46,21,0.486667,0.502340,0.142857
9,Baker's,46,10,0.623611,0.334962,0.000000


Pseudo-label probability summary:


,mean,std,min,max
ACL,0.412167,0.036959,0.314111,0.512109
MCL,0.193740,0.032641,0.113654,0.316626
Medial Meniscus,0.419755,0.079727,0.235449,0.559375
Lateral Meniscus,0.377397,0.047854,0.277417,0.528564
Medial OA,0.280575,0.045537,0.167920,0.421484
Lateral OA,0.222300,0.032412,0.144202,0.377197
PF OA,0.359352,0.047457,0.271582,0.494922
Effusion,0.598542,0.035679,0.513965,0.666406
Synovitis,0.455833,0.031581,0.378418,0.533936
Baker's,0.234939,0.043300,0.128705,0.351807


Mean five-fold disagreement:


,mean_fold_std
ACL,0.060223
MCL,0.056711
Medial Meniscus,0.068225
Lateral Meniscus,0.054662
Medial OA,0.046252
Lateral OA,0.057320
PF OA,0.067431
Effusion,0.044541
Synovitis,0.037067
Baker's,0.054034


## 7. DICOM loading and MRNet-style model

Slices are ordered by physical geometry when possible, then `InstanceNumber`, then filename. Intensities use DICOM slope/intercept, MONOCHROME1 inversion, whole-series 1st/99th percentile clipping, aspect-preserving resize, and ImageNet normalization.

MRNet processes every selected slice independently with AlexNet features, global-pools each slice, and performs element-wise max pooling across the 10 sampled slices. It repeats this operation for all three series, then performs a second element-wise max across the three series representations before mapping the resulting 256-dimensional study representation to 12 logits. The loss is ordinary BCE with the **soft KBioXLM ensemble probabilities**; fold disagreement is saved but not used for filtering or weighting in this first experiment.

In [9]:
def visible_files(folder):
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith("."))

def geometric_scalar(dcm):
    ori = getattr(dcm, "ImageOrientationPatient", None)
    pos = getattr(dcm, "ImagePositionPatient", None)
    if ori is None or pos is None:
        return None
    try:
        ori = np.asarray(ori, dtype=float)
        pos = np.asarray(pos, dtype=float)
        normal = np.cross(ori[:3], ori[3:])
        norm = np.linalg.norm(normal)
        return None if norm == 0 else float(np.dot(pos, normal / norm))
    except Exception:
        return None

def ordered_dicom_paths(series_dir):
    records = []
    for path in visible_files(series_dir):
        try:
            dcm = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        except Exception:
            continue
        geom = geometric_scalar(dcm)
        instance = getattr(dcm, "InstanceNumber", None)
        try:
            instance = float(instance) if instance is not None else None
        except Exception:
            instance = None
        records.append((path, geom, instance))
    if records and all(r[1] is not None for r in records):
        records.sort(key=lambda r: r[1])
    elif records and all(r[2] is not None for r in records):
        records.sort(key=lambda r: r[2])
    else:
        records.sort(key=lambda r: r[0].name)
    return [r[0] for r in records]

def resize_pad(image, size=IMAGE_SIZE):
    h, w = image.shape
    scale = size / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size), dtype=np.float32)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas

def decode_dicom(path):
    dcm = pydicom.dcmread(str(path), force=True)
    x = dcm.pixel_array.astype(np.float32)
    if x.ndim > 2:
        x = np.squeeze(x)
    if x.ndim != 2:
        raise ValueError(f"Unsupported pixel shape {x.shape}")
    x = x * float(getattr(dcm, "RescaleSlope", 1.0)) + float(getattr(dcm, "RescaleIntercept", 0.0))
    if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
        x = x.max() - x
    return x

def load_series_volume(series_root, study_id, series_id, max_slices=MAX_SLICES):
    paths = ordered_dicom_paths(series_root / str(study_id) / str(series_id))
    if not paths:
        raise ValueError(f"No readable DICOM headers for {study_id}/{series_id}")

    # Decode near evenly spaced targets. If a target is corrupt, try nearby slices.
    targets = np.linspace(0, len(paths) - 1, max_slices).round().astype(int)
    cache = {}
    raw = []
    for target in targets:
        chosen = None
        offsets = [0]
        for distance in range(1, len(paths)):
            offsets.extend([-distance, distance])
        for offset in offsets:
            idx = int(target + offset)
            if idx < 0 or idx >= len(paths):
                continue
            if idx not in cache:
                try:
                    cache[idx] = decode_dicom(paths[idx])
                except Exception:
                    cache[idx] = None
            if cache[idx] is not None:
                chosen = cache[idx]
                break
        if chosen is not None:
            raw.append(chosen)

    if not raw:
        raise ValueError(f"No decodable pixel data for {study_id}/{series_id}")
    if len(raw) != max_slices:
        indices = np.linspace(0, len(raw) - 1, max_slices).round().astype(int)
        raw = [raw[i] for i in indices]

    finite_chunks = [x[np.isfinite(x)] for x in raw if np.isfinite(x).any()]
    if not finite_chunks:
        raise ValueError(f"No finite pixels for {study_id}/{series_id}")
    lo, hi = np.percentile(np.concatenate(finite_chunks), [1, 99])
    processed = []
    for x in raw:
        x = np.nan_to_num(x, nan=lo, posinf=hi, neginf=lo)
        x = (np.clip(x, lo, hi) - lo) / (hi - lo) if hi > lo else np.zeros_like(x)
        processed.append(resize_pad(x))
    return torch.from_numpy(np.stack(processed)[:, None].astype(np.float32))

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class MRNet12(nn.Module):
    def __init__(self, checkpoint_path=None):
        super().__init__()
        base = alexnet(weights=None)
        if checkpoint_path is not None:
            try:
                state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
            except TypeError:
                state = torch.load(checkpoint_path, map_location="cpu")
            base.load_state_dict(state)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(256, len(LABEL_COLS))
        for p in self.features.parameters():
            p.requires_grad = False
        for layer in list(self.features.children())[6:]:
            for p in layer.parameters():
                p.requires_grad = True

    def encode_series(self, volume):
        chunks = []
        mean = IMAGENET_MEAN.to(volume.device)
        std = IMAGENET_STD.to(volume.device)
        for start in range(0, len(volume), SLICE_CHUNK):
            x = volume[start:start + SLICE_CHUNK].repeat(1, 3, 1, 1)
            x = (x - mean) / std
            f = self.pool(self.features(x)).flatten(1)
            chunks.append(f)
        return torch.cat(chunks, dim=0).max(dim=0).values

    def forward(self, volumes):
        if not 1 <= len(volumes) <= SERIES_PER_STUDY:
            raise ValueError(f"Expected one to {SERIES_PER_STUDY} series, received {len(volumes)}")
        series_features = torch.stack([self.encode_series(volume) for volume in volumes], dim=0)
        study_features = series_features.max(dim=0).values
        return self.head(self.dropout(study_features))


## 8. Assemble the complete 1,000-study MRI training pool

All 46 development-gold studies are now included in MRI training with their true binary targets. The 954 disjoint report-labelled studies retain the five-teacher ensemble probabilities as soft targets. The untouched 12-study gold holdout is never used for report training, MRI training, stopping, or model selection.

Because the complete 1,000-study pool is intentionally used for optimization, MRI training runs for four fixed epochs selected before viewing the final holdout. This avoids silently withholding approximately 10% of the requested pool for weak-label early stopping.


In [10]:
soft_targets = weak_labels[[f"weak__{x}" for x in LABEL_COLS]].to_numpy(np.float32)

# Convert the 46 development-gold labels into the same target schema used by the
# pseudo-labelled studies. Gold targets remain exact 0/1 values.
gold_image_labels = gold_dev[["StudyInstanceUID"] + LABEL_COLS].copy()
for label in LABEL_COLS:
    gold_image_labels[f"weak__{label}"] = gold_image_labels[label].astype(np.float32)
    gold_image_labels[f"std__{label}"] = 0.0
gold_image_labels = gold_image_labels[
    ["StudyInstanceUID"]
    + [f"weak__{label}" for label in LABEL_COLS]
    + [f"std__{label}" for label in LABEL_COLS]
]
gold_image_labels["target_source"] = "gold"

weak_image_labels = weak_labels.copy()
weak_image_labels["target_source"] = "report_soft"

# Select three readable series for the 46 development-gold studies using the
# same acquisition-frequency ordering learned from the non-gold training pool.
gold_series_candidates = series_meta[
    series_meta["StudyInstanceUID"].astype(str).isin(set(gold_dev.StudyInstanceUID.astype(str)))
].copy()
gold_series_ok = []
for row in gold_series_candidates.itertuples(index=False):
    directory = TRAIN_SERIES_ROOT / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID)
    try:
        ok = directory.exists() and any(
            path.is_file() and not path.name.startswith(".")
            for path in directory.iterdir()
        )
    except Exception:
        ok = False
    gold_series_ok.append(ok)
gold_series_candidates = gold_series_candidates.loc[gold_series_ok].copy()
gold_series_candidates["signature_rank"] = (
    gold_series_candidates["signature"].map(signature_rank).fillna(len(signature_rank) + 1)
)
gold_series_candidates["target_first"] = (
    gold_series_candidates["signature"] != TARGET_SIGNATURE
).astype(int)

gold_series_selection = (
    gold_series_candidates
    .drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])
    .sort_values(["StudyInstanceUID", "target_first", "signature_rank", "SeriesInstanceUID"])
    .groupby("StudyInstanceUID", group_keys=False)
    .head(SERIES_PER_STUDY)
    .reset_index(drop=True)
)
gold_series_counts = gold_series_selection.groupby("StudyInstanceUID").size()
complete_gold_ids = set(gold_series_counts[gold_series_counts == SERIES_PER_STUDY].index.astype(str))
missing_gold_ids = set(gold_dev.StudyInstanceUID.astype(str)) - complete_gold_ids
if missing_gold_ids:
    raise RuntimeError(
        "Development-gold studies without three readable training series: "
        f"{sorted(missing_gold_ids)}"
    )

image_train_df = pd.concat(
    [weak_image_labels, gold_image_labels],
    ignore_index=True,
    sort=False,
)
image_series_selection = pd.concat(
    [weak_series_selection, gold_series_selection],
    ignore_index=True,
    sort=False,
).drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])

if len(weak_image_labels) != 954:
    warnings.warn(f"Expected 954 soft-labelled studies, found {len(weak_image_labels)}")
if len(image_train_df) != len(weak_image_labels) + len(gold_image_labels):
    raise RuntimeError("MRI training-pool assembly changed the study count")
if set(gold_final.StudyInstanceUID.astype(str)) & set(image_train_df.StudyInstanceUID.astype(str)):
    raise RuntimeError("Final-gold holdout leaked into MRI training")

print("MRI training studies:", len(image_train_df))
print("  true-gold:", int((image_train_df.target_source == "gold").sum()))
print("  report-soft:", int((image_train_df.target_source == "report_soft").sum()))
print("Selected MRI series:", len(image_series_selection))

class StudyDataset(Dataset):
    def __init__(self, frame, selected_series, series_root, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.series_map = {
            str(study_id): group.sort_values(["target_first", "signature_rank", "SeriesInstanceUID"])
            for study_id, group in selected_series.groupby("StudyInstanceUID")
        }
        self.series_root = series_root
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        study_id = str(row.StudyInstanceUID)
        selected = self.series_map[study_id]
        if len(selected) != SERIES_PER_STUDY:
            raise ValueError(f"Study {study_id} has {len(selected)} selected series")
        volumes = [
            load_series_volume(self.series_root, study_id, series_id)
            for series_id in selected.SeriesInstanceUID.astype(str)
        ]
        if self.augment and np.random.random() < 0.5:
            volumes = [torch.flip(volume, dims=[-1]) for volume in volumes]
        target = torch.tensor(
            [row[f"weak__{label}"] for label in LABEL_COLS],
            dtype=torch.float32,
        )
        return {
            "volumes": volumes,
            "target": target,
            "row_index": idx,
            "target_source": row["target_source"],
        }

def collate_study(batch):
    return batch[0]

train_image_loader = DataLoader(
    StudyDataset(image_train_df, image_series_selection, TRAIN_SERIES_ROOT, augment=True),
    batch_size=1,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=AMP,
    collate_fn=collate_study,
)


MRI training studies: 1000
  true-gold: 46
  report-soft: 954
Selected MRI series: 3000


In [11]:
seed_everything(SEED + 500)
image_model = MRNet12(ALEXNET_WEIGHTS).to(DEVICE)
backbone_params = [p for p in image_model.features.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": IMAGE_LR_BACKBONE},
    {"params": image_model.head.parameters(), "lr": IMAGE_LR_HEAD},
], weight_decay=IMAGE_WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)
criterion = nn.BCEWithLogitsLoss()

image_history = []
for epoch in range(1, IMAGE_EPOCHS + 1):
    image_model.train()
    optimizer.zero_grad(set_to_none=True)
    train_losses = []
    gold_losses = []
    soft_losses = []

    for step, batch in enumerate(train_image_loader, 1):
        volumes = [volume.to(DEVICE, non_blocking=True) for volume in batch["volumes"]]
        target = batch["target"].to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            logits = image_model(volumes)
            raw_loss = criterion(logits, target)
            loss = raw_loss / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_image_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                [p for p in image_model.parameters() if p.requires_grad],
                1.0,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        value = float(raw_loss.detach().cpu())
        train_losses.append(value)
        if batch["target_source"] == "gold":
            gold_losses.append(value)
        else:
            soft_losses.append(value)

    row = {
        "epoch": epoch,
        "train_bce": float(np.mean(train_losses)),
        "gold_bce": float(np.mean(gold_losses)) if gold_losses else np.nan,
        "soft_bce": float(np.mean(soft_losses)) if soft_losses else np.nan,
    }
    image_history.append(row)
    print(
        f"epoch {epoch:02d}/{IMAGE_EPOCHS} | all {row['train_bce']:.4f} "
        f"| gold {row['gold_bce']:.4f} | soft {row['soft_bce']:.4f}"
    )

best_image_state = {
    key: value.detach().cpu().clone()
    for key, value in image_model.state_dict().items()
}
image_model.load_state_dict(best_image_state)
image_model.eval()

pd.DataFrame(image_history).to_csv(OUT / "mrnet_training_history.csv", index=False)
torch.save(best_image_state, OUT / "mrnet_1000studies_3series_10slices_final.pt")
display(pd.DataFrame(image_history))

# Save a self-contained diagnostic/reuse bundle in addition to submission.csv.
fallback_probabilities = torch.tensor(soft_targets.mean(axis=0), dtype=torch.float32)
bundle = {
    "format_version": 2,
    "architecture": "alexnet_mrnet12",
    "state_dict": best_image_state,
    "label_cols": list(LABEL_COLS),
    "fallback_probabilities": fallback_probabilities,
    "target_signature": str(TARGET_SIGNATURE),
    "signature_rank": {str(k): int(v) for k, v in signature_rank.items()},
    "image_size": int(IMAGE_SIZE),
    "max_slices": int(MAX_SLICES),
    "series_per_study": int(SERIES_PER_STUDY),
    "n_gold_train": int(len(gold_image_labels)),
    "n_soft_train": int(len(weak_image_labels)),
}
torch.save(bundle, OUT / "rsna_mrnet_1000study_inference_bundle.pt")


epoch 01/4 | all 0.6528 | gold 0.6555 | soft 0.6526
epoch 02/4 | all 0.6232 | gold 0.6107 | soft 0.6238
epoch 03/4 | all 0.6209 | gold 0.6097 | soft 0.6215
epoch 04/4 | all 0.6196 | gold 0.5983 | soft 0.6207


,epoch,train_bce,gold_bce,soft_bce
0,1,0.652773,0.655472,0.652643
1,2,0.623170,0.610732,0.623769
2,3,0.620941,0.609739,0.621481
3,4,0.619642,0.598251,0.620674


## 9. One-time evaluation on the untouched 12-study gold holdout

For each study, the notebook selects exactly three readable series using the same acquisition-frequency ranking used for weak training. No holdout result is used to retrain or select the model.

In [12]:
def attach_signatures(frame):
    x = frame.copy()
    for col in ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]:
        if col not in x.columns: x[col] = "UNKNOWN"
    x["plane_norm"] = x["Anatomical_Plane"].map(canonical_text).replace({"SAG":"SAGITTAL", "COR":"CORONAL", "AX":"AXIAL"})
    x["fluid_norm"] = x["Fluid_Sensitive"].map(boolish_text)
    x["fat_norm"] = x["Fat_Suppression"].map(boolish_text)
    x["signature"] = x[["plane_norm", "fluid_norm", "fat_norm"]].agg("|".join, axis=1)
    x["signature_rank"] = x["signature"].map(signature_rank).fillna(len(signature_rank) + 1)
    x["target_first"] = (x["signature"] != TARGET_SIGNATURE).astype(int)
    return x

def select_three_series_per_study(series_frame, study_ids, series_root):
    x = attach_signatures(series_frame)
    x = x[x.StudyInstanceUID.astype(str).isin(set(map(str, study_ids)))].copy()
    ok = []
    for r in x.itertuples(index=False):
        d = series_root / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
        ok.append(d.exists() and any(p.is_file() and not p.name.startswith(".") for p in d.iterdir()))
    x = x.loc[ok]
    x = x.sort_values(["StudyInstanceUID", "target_first", "signature_rank", "SeriesInstanceUID"])
    x = x.drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])
    return x.groupby("StudyInstanceUID", group_keys=False).head(SERIES_PER_STUDY).reset_index(drop=True)

@torch.inference_mode()
def predict_selected_studies(selected, series_root):
    records = []
    groups = list(selected.groupby("StudyInstanceUID", sort=True))
    for i, (study_id, group) in enumerate(groups, 1):
        if len(group) != SERIES_PER_STUDY:
            raise ValueError(f"Study {study_id} has {len(group)} readable selected series")
        volumes = [
            load_series_volume(series_root, study_id, series_id).to(DEVICE)
            for series_id in group.SeriesInstanceUID.astype(str)
        ]
        with torch.cuda.amp.autocast(enabled=AMP):
            prob = torch.sigmoid(image_model(volumes)).float().cpu().numpy()
        rec = {
            "StudyInstanceUID": str(study_id),
            "SeriesInstanceUIDs": "|".join(group.SeriesInstanceUID.astype(str)),
            "n_target_signature_series": int((group.signature == TARGET_SIGNATURE).sum()),
        }
        rec.update({f"pred__{label}": float(prob[j]) for j, label in enumerate(LABEL_COLS)})
        records.append(rec)
        if i % 25 == 0 or i == len(groups):
            print(f"Predicted {i}/{len(groups)} studies")
    return pd.DataFrame(records)

final_selected = select_three_series_per_study(
    train_series, gold_final.StudyInstanceUID, TRAIN_SERIES_ROOT
)
final_counts = final_selected.groupby("StudyInstanceUID").size()
complete_final_ids = set(final_counts[final_counts == SERIES_PER_STUDY].index)
missing_final = set(gold_final.StudyInstanceUID) - complete_final_ids
if missing_final:
    raise RuntimeError(f"Final holdout studies without three readable series: {sorted(missing_final)}")

final_pred = predict_selected_studies(final_selected, TRAIN_SERIES_ROOT)
final_eval = gold_final[["StudyInstanceUID"] + LABEL_COLS].merge(final_pred, on="StudyInstanceUID", how="inner")
y_final = final_eval[LABEL_COLS].to_numpy(np.float32)
p_final = final_eval[[f"pred__{x}" for x in LABEL_COLS]].to_numpy(np.float32)
final_metrics = multilabel_metrics(y_final, p_final)
final_metrics.to_csv(OUT / "final_gold_12_metrics.csv", index=False)
final_eval.to_csv(OUT / "final_gold_12_predictions.csv", index=False)
print("Final holdout studies evaluated with exactly three series:", len(final_pred))
display(final_metrics)

Predicted 12/12 studies
Final holdout studies evaluated with exactly three series: 12


,pathology,n,positives,roc_auc,pr_auc,f1_at_0.5
0,ACL,12,5,0.485714,0.458889,0.000000
1,MCL,12,2,0.450000,0.208333,0.000000
2,Medial Meniscus,12,6,0.750000,0.691667,0.000000
3,Lateral Meniscus,12,5,0.457143,0.537937,0.000000
4,Medial OA,12,3,0.629630,0.369444,0.000000
5,Lateral OA,12,2,0.100000,0.138889,0.000000
6,PF OA,12,5,0.314286,0.376667,0.000000
7,Effusion,12,7,0.800000,0.904762,0.736842
8,Synovitis,12,6,0.555556,0.692593,0.000000
9,Baker's,12,2,0.200000,0.154762,0.000000


## 10. Hidden-test inference and exact `submission.csv`

The test study IDs and final column order come only from `sample_submission.csv`. The code checks both Kaggle competition mount conventions and the common `test_series`, `test`, `test_images`, and substituted `train_series` layouts.

Each hidden study starts with the per-pathology training-target prevalence as a safe fallback. Successfully decoded predictions overwrite it. Individual series or study failures are logged to `test_inference_failures.csv` without terminating the rerun.


In [13]:
def candidate_test_roots():
    candidates = [
        ROOT / "test_series",
        ROOT / "test",
        ROOT / "test_images",
        TRAIN_SERIES_ROOT,
    ]
    return [path for path in candidates if path.exists() and path.is_dir()]

def choose_test_series_root(study_ids):
    ids = [str(x) for x in study_ids]
    candidates = candidate_test_roots()
    # Prefer the directory that actually contains one of the submitted study IDs.
    for directory in candidates:
        if any((directory / sid).is_dir() for sid in ids[: min(25, len(ids))]):
            return directory
    # Public placeholder IDs may have no public image directories. Keep a deterministic
    # existing root so that a valid fallback submission can still be produced.
    return candidates[0] if candidates else None

def load_test_series_metadata(test_root, study_ids):
    ids = set(map(str, study_ids))
    csv_candidates = [
        ROOT / "test_series.csv",
        ROOT / "series.csv",
        ROOT / "train_series.csv",
    ]
    for csv_path in csv_candidates:
        if not csv_path.exists():
            continue
        frame = pd.read_csv(
            csv_path,
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        ).drop_duplicates()
        overlap = frame["StudyInstanceUID"].astype(str).isin(ids).any()
        if overlap:
            print("Using test-series metadata:", csv_path)
            return frame

    # Metadata-free fallback: enumerate study/series folders from sample IDs.
    rows = []
    if test_root is not None:
        for study_id in sorted(ids):
            study_dir = test_root / study_id
            if not study_dir.exists():
                continue
            try:
                series_dirs = sorted(path for path in study_dir.iterdir() if path.is_dir())
            except Exception:
                series_dirs = []
            for series_dir in series_dirs:
                rows.append({
                    "StudyInstanceUID": study_id,
                    "SeriesInstanceUID": series_dir.name,
                    "Anatomical_Plane": "UNKNOWN",
                    "Fluid_Sensitive": "UNKNOWN",
                    "Fat_Suppression": "UNKNOWN",
                })
    print("Enumerated test-series metadata rows:", len(rows))
    return pd.DataFrame(rows, columns=[
        "StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane",
        "Fluid_Sensitive", "Fat_Suppression",
    ])

sample_submission = pd.read_csv(
    ROOT / "sample_submission.csv",
    dtype={"StudyInstanceUID": str},
)
if "StudyInstanceUID" not in sample_submission.columns:
    raise ValueError("sample_submission.csv has no StudyInstanceUID column")

submission_labels = [column for column in sample_submission.columns if column != "StudyInstanceUID"]
missing_outputs = [column for column in submission_labels if column not in LABEL_COLS]
if missing_outputs:
    raise ValueError(f"Model has no output for submission columns: {missing_outputs}")

test_ids = sample_submission["StudyInstanceUID"].astype(str).tolist()
TEST_SERIES_ROOT = choose_test_series_root(test_ids)
test_series = load_test_series_metadata(TEST_SERIES_ROOT, test_ids)

if TEST_SERIES_ROOT is None:
    print("SEVERE WARNING: no MRI series root was found; all rows will use fallback probabilities")
else:
    print("Test MRI root:", TEST_SERIES_ROOT)

test_selected = select_three_series_per_study(
    test_series,
    test_ids,
    TEST_SERIES_ROOT,
) if TEST_SERIES_ROOT is not None else pd.DataFrame()

fallback = np.asarray(soft_targets.mean(axis=0), dtype=np.float32)
pred_map = {study_id: fallback.copy() for study_id in test_ids}
failures = []

image_model.eval()
for index, study_id in enumerate(test_ids, 1):
    if test_selected.empty:
        group = pd.DataFrame()
    else:
        group = test_selected[
            test_selected["StudyInstanceUID"].astype(str) == study_id
        ]

    volumes = []
    errors = []
    if not group.empty:
        for series_id in group["SeriesInstanceUID"].astype(str):
            try:
                volume = load_series_volume(
                    TEST_SERIES_ROOT,
                    study_id,
                    series_id,
                    max_slices=MAX_SLICES,
                ).to(DEVICE)
                volumes.append(volume)
            except Exception as exc:
                errors.append(f"{series_id}: {type(exc).__name__}: {exc}")

    if volumes:
        try:
            with torch.inference_mode(), torch.cuda.amp.autocast(enabled=AMP):
                probabilities = torch.sigmoid(image_model(volumes)).float().cpu().numpy()
            probabilities = np.where(np.isfinite(probabilities), probabilities, fallback)
            pred_map[study_id] = np.clip(probabilities, 0.0, 1.0)
        except Exception as exc:
            errors.append(f"model: {type(exc).__name__}: {exc}")
    else:
        errors.append("No decodable selected MRI series; used per-label fallback")

    if errors:
        failures.append({
            "StudyInstanceUID": study_id,
            "n_selected_series": int(len(group)),
            "n_decoded_series": int(len(volumes)),
            "error": " || ".join(errors),
        })
    if index % 25 == 0 or index == len(test_ids):
        print(f"Test inference {index}/{len(test_ids)}")

submission = sample_submission[["StudyInstanceUID"]].copy()
prediction_matrix = np.stack([pred_map[study_id] for study_id in test_ids])
for label in submission_labels:
    submission[label] = prediction_matrix[:, LABEL_COLS.index(label)]

values = submission[submission_labels].to_numpy(dtype=np.float64)
values = np.clip(np.nan_to_num(values, nan=0.5, posinf=1.0, neginf=0.0), 0.0, 1.0)
submission.loc[:, submission_labels] = values
submission = submission.reindex(columns=sample_submission.columns)

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
FAILURE_PATH = OUT / "test_inference_failures.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
pd.DataFrame(
    failures,
    columns=["StudyInstanceUID", "n_selected_series", "n_decoded_series", "error"],
).to_csv(FAILURE_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
print("Submission shape:", submission.shape)
print("Studies with any recorded issue:", len(failures))
print("Studies using complete fallback:", sum(row["n_decoded_series"] == 0 for row in failures))
display(submission.head())


Using test-series metadata: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv
Test MRI root: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
Test inference 3/3
Saved: /kaggle/working/submission.csv
Submission shape: (3, 13)
Studies with any recorded issue: 0
Studies using complete fallback: 0


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.435791,0.198975,0.453857,0.386719,0.252197,0.227173,0.345215,0.626465,0.464355,0.214478,0.299805,0.315430
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.416260,0.199463,0.470215,0.389404,0.292236,0.211304,0.357422,0.604492,0.437500,0.258545,0.327393,0.280029
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.424561,0.203491,0.411377,0.392578,0.297363,0.231934,0.365479,0.627930,0.458984,0.246582,0.318848,0.276123


## Submission checklist

1. Attach the competition data and your `rsna-offline-models` Kaggle Model.
2. Keep Internet disabled and enable a GPU accelerator.
3. Choose **Save Version → Save & Run All**.
4. Confirm the final cell prints `Saved: /kaggle/working/submission.csv`.
5. The public placeholder run may legitimately use fallbacks if its sample IDs have no public MRI directories. The private rerun substitutes the real test data.
6. Select `/kaggle/working/submission.csv` as the competition output file.

Research diagnostics, the trained checkpoint, teacher metrics, holdout metrics, and inference failures are also saved under `/kaggle/working/rsna_kbioxlm_mrnet/`.
